# Trust Region Policy Optimisation (TRPO)

## Background

NPG (notebook 08) applies the natural gradient with a fixed step size.
If the step is too large the policy can change catastrophically and
performance collapses. TRPO (Schulman et al. 2015) adds a safety constraint:

```
maximise  J(pi_new) - J(pi_old)
subject to  KL( pi_old || pi_new ) <= delta
```

The KL constraint bounds how much the policy is allowed to change in one step.

## Algorithm for tabular softmax

1. **Compute advantage** A(s,a) = Q(s,a) - V(s) from a Monte-Carlo rollout.
2. **Natural gradient direction** d[s, :] = A(s, :) (closed form, see nb 08).
3. **Maximum step size** via the quadratic KL approximation:
   ```
   KL(pi_old || pi_{old + alpha*d}) approx (alpha^2 / 2) * d^T F d
   -> alpha_max = sqrt(2 * delta / (d^T F d))
   ```
   where `d^T F_s d = Var_{pi_s}[d_s(.)]` for each state s.
4. **Backtracking line search**: shrink alpha until exact KL <= delta.

For the tabular case the Fisher-vector product has a closed form:
```
F_s d_s = diag(pi_s) * d_s - pi_s * (pi_s^T d_s)
        = pi_s * (d_s - V_s * 1)    where V_s = pi_s^T d_s = 0 for advantage
        = pi_s * A_s
```
So `d_s^T F_s d_s = sum_a pi_s(a) * A_s(a)^2`  (variance of A under pi_s).

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from environment.gridworld import GridWorldEnv
from experiments.run_all import load_config
from utils.plotting import (
    plot_multi_curves, plot_value_heatmap, plot_vstar_heatmap,
    plot_policy_arrows, plot_summary_bar, save_figure,
)
from utils.metrics import signed_error, abs_error, policy_eval_error

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

cfg    = load_config()
N_EPS  = cfg['environment']['n_episodes']
MAX_ST = cfg['environment']['max_steps_per_episode']
GAMMA  = cfg['environment']['gamma']

def flat(obs):
    return int(obs[0]) * 8 + int(obs[1])

def _softmax(t: np.ndarray) -> np.ndarray:
    e = np.exp(t - t.max()); return e / e.sum()

## 1. KL Divergence and Fisher-Vector Product

We verify the closed-form expressions before building the agent.

In [ ]:
def kl_divergence_state(theta_old_s: np.ndarray, theta_new_s: np.ndarray) -> float:
    """KL(pi_old || pi_new) for one state. Uses log-sum-exp for stability."""
    pi_old = _softmax(theta_old_s)
    pi_new = _softmax(theta_new_s)
    return float(np.sum(pi_old * (np.log(pi_old + 1e-12) - np.log(pi_new + 1e-12))))


def fisher_vector_product(pi_s: np.ndarray, v_s: np.ndarray) -> np.ndarray:
    """
    F_s @ v_s  where F_s = diag(pi_s) - pi_s * pi_s^T.

    Computed in O(n) without materialising the matrix:
        F_s v_s = pi_s * v_s - pi_s * (pi_s . v_s)
    """
    return pi_s * v_s - pi_s * float(np.dot(pi_s, v_s))


def vFv(pi_s: np.ndarray, v_s: np.ndarray) -> float:
    """v^T F_s v = variance of v under pi_s."""
    return float(np.dot(v_s, fisher_vector_product(pi_s, v_s)))


# Quick sanity check
rng  = np.random.default_rng(0)
pi_s = _softmax(rng.standard_normal(9))
A_s  = rng.standard_normal(9)
A_s -= float(np.dot(pi_s, A_s))   # centre: sum pi*A = 0

# Quadratic KL approximation for small step alpha
alpha = 0.01
theta_old_s = np.log(pi_s + 1e-12)
theta_new_s = theta_old_s + alpha * A_s
kl_exact    = kl_divergence_state(theta_old_s, theta_new_s)
kl_approx   = 0.5 * alpha**2 * vFv(pi_s, A_s)
print(f'KL exact   = {kl_exact:.6f}')
print(f'KL approx  = {kl_approx:.6f}  (should match for small alpha)')

## 2. TRPO Agent

In [ ]:
class TRPOAgent:
    """
    Trust Region Policy Optimisation for tabular softmax.

    Per episode:
      1. Estimate Q(s,a) via running Monte-Carlo mean.
      2. Compute advantage A(s,.) = Q(s,.) - V(s)  (natural gradient direction).
      3. Compute max step alpha from quadratic KL approximation:
             alpha_max = sqrt(2 * delta / sum_s w_s * A_s^T F_s A_s)
         where w_s = visit_count_s / total_steps.
      4. Backtracking line search: shrink alpha by 0.5 until exact KL <= delta.
      5. Apply: theta[s, :] += alpha * A(s, :) for visited states.

    Records step sizes and KL values for analysis.
    """

    def __init__(self, rng, gamma: float, delta: float = 0.01,
                 max_alpha: float = 1.0, line_search_steps: int = 15,
                 n_states: int = 64, n_actions: int = 9):
        self._rng = rng
        self.gamma = gamma
        self.delta = delta
        self.max_alpha = max_alpha
        self.line_search_steps = line_search_steps
        self.n_states = n_states
        self.n_actions = n_actions
        self.theta  = np.zeros((n_states, n_actions))
        self._Q_hat = np.zeros((n_states, n_actions))
        self._Q_cnt = np.zeros((n_states, n_actions))
        self._buf   = []
        self.kl_history    = []   # KL achieved each episode
        self.alpha_history = []   # step size accepted each episode

    def select_action(self, s: int) -> int:
        return int(self._rng.choice(self.n_actions, p=_softmax(self.theta[s, :])))

    def reset_episode(self): self._buf = []

    def update(self, s, a, cost, s_prime, done):
        self._buf.append((s, a, float(cost)))

    def _mean_kl(self, theta_old, theta_new, visited_states, visit_wts):
        """Weighted mean KL over visited states."""
        kl = 0.0
        for s in visited_states:
            kl += visit_wts[s] * kl_divergence_state(theta_old[s, :], theta_new[s, :])
        return kl

    def finish_episode(self):
        if not self._buf:
            return

        # --- step 1: Monte-Carlo returns ---
        G = 0.0
        ep_Q: dict[tuple, float] = {}
        for s, a, cost in reversed(self._buf):
            G = cost + self.gamma * G
            ep_Q[(s, a)] = G

        for (s, a), G_t in ep_Q.items():
            self._Q_cnt[s, a] += 1
            self._Q_hat[s, a] += (G_t - self._Q_hat[s, a]) / self._Q_cnt[s, a]

        # --- step 2: advantage and direction ---
        visited = list(set(s for s, a, _ in self._buf))
        T = len(self._buf)
        visit_cnt  = {s: sum(1 for ss, _, _ in self._buf if ss == s) for s in visited}
        visit_wts  = {s: visit_cnt[s] / T for s in visited}

        direction = np.zeros_like(self.theta)   # natural gradient direction
        for s in visited:
            pi_s = _softmax(self.theta[s, :])
            V_s  = float(np.dot(pi_s, self._Q_hat[s, :]))
            direction[s, :] = self._Q_hat[s, :] - V_s   # A(s, .)

        # --- step 3: max step from quadratic KL approximation ---
        dFd = sum(
            visit_wts[s] * vFv(_softmax(self.theta[s, :]), direction[s, :])
            for s in visited
        )
        if dFd < 1e-12:
            self.kl_history.append(0.0)
            self.alpha_history.append(0.0)
            return

        alpha = min(self.max_alpha, float(np.sqrt(2.0 * self.delta / dFd)))

        # --- step 4: backtracking line search ---
        theta_old = self.theta.copy()
        accepted  = False
        for _ in range(self.line_search_steps):
            theta_try = theta_old.copy()
            for s in visited:
                theta_try[s, :] += alpha * direction[s, :]
            kl = self._mean_kl(theta_old, theta_try, visited, visit_wts)
            if kl <= self.delta:
                self.theta = theta_try
                self.kl_history.append(kl)
                self.alpha_history.append(alpha)
                accepted = True
                break
            alpha *= 0.5

        if not accepted:
            self.kl_history.append(0.0)
            self.alpha_history.append(0.0)

    def get_policy(self) -> np.ndarray:
        return np.vstack([_softmax(self.theta[s, :]) for s in range(self.n_states)])

    def get_value_estimate(self) -> np.ndarray:
        V = np.zeros(self.n_states)
        for s in range(self.n_states):
            V[s] = float(np.dot(_softmax(self.theta[s, :]), self._Q_hat[s, :]))
        return V

## 3. Experiments

In [ ]:
def run_one_episode(agent, env, max_steps):
    agent.reset_episode()
    obs = env.reset(); s = flat(obs)
    for _ in range(max_steps):
        a = agent.select_action(s)
        obs, cost, done, _ = env.step(a)
        s_prime = flat(obs)
        agent.update(s, a, cost, s_prime, done)
        s = s_prime
        if done: break
    agent.finish_episode()


def multi_run(factory, n_runs=3, n_eps=N_EPS, max_steps=MAX_ST,
              eval_every=50, base_seed=0) -> dict:
    ckpt_eps   = np.arange(eval_every - 1, n_eps, eval_every)
    signed_arr = np.zeros((n_runs, n_eps))
    abs_arr    = np.zeros((n_runs, n_eps))
    policy_arr = np.zeros((n_runs, len(ckpt_eps)))
    for run in range(n_runs):
        rng_a = np.random.default_rng(base_seed + run * 1000)
        rng_e = np.random.default_rng(base_seed + run * 1000 + 1)
        agent = factory(rng_a); env = GridWorldEnv(rng_e)
        ci = 0
        for ep in range(n_eps):
            run_one_episode(agent, env, max_steps)
            signed_arr[run, ep] = signed_error(agent.get_value_estimate())
            abs_arr[run, ep]    = abs_error(agent.get_value_estimate())
            if ci < len(ckpt_eps) and ep == ckpt_eps[ci]:
                policy_arr[run, ci] = policy_eval_error(agent.get_policy()); ci += 1
        while ci < len(ckpt_eps):
            policy_arr[run, ci] = policy_eval_error(agent.get_policy()); ci += 1
    return {'signed_err': signed_arr, 'abs_err': abs_arr,
            'policy_err': policy_arr, 'checkpoint_eps': ckpt_eps,
            'n_runs': n_runs, 'n_episodes': n_eps}

In [ ]:
N_RUNS = 3
DELTA  = 0.01     # KL trust region radius

print('TRPO (delta=0.01) ...')
trpo_results = multi_run(
    lambda rng: TRPOAgent(rng, GAMMA, delta=DELTA), N_RUNS)

# NPG reference: same direction, no constraint
from notebooks_utils_local import NaturalPGAgent  # if already defined above; otherwise inline
# If the cell above defining NaturalPGAgent is not in scope, redefine here:

class NaturalPGRef:
    """NPG reference — same advantage update without KL constraint."""
    def __init__(self, rng, gamma, lr=0.01, n_states=64, n_actions=9):
        self._rng, self.gamma, self.lr = rng, gamma, lr
        self.n_states, self.n_actions = n_states, n_actions
        self.theta = np.zeros((n_states, n_actions))
        self._Q_hat = np.zeros((n_states, n_actions))
        self._Q_cnt = np.zeros((n_states, n_actions))
        self._buf = []
    def select_action(self, s):
        return int(self._rng.choice(self.n_actions, p=_softmax(self.theta[s, :])))
    def reset_episode(self): self._buf = []
    def update(self, s, a, cost, s_prime, done): self._buf.append((s, a, float(cost)))
    def finish_episode(self):
        G = 0.0; ep_Q = {}
        for s, a, cost in reversed(self._buf):
            G = cost + self.gamma * G; ep_Q[(s, a)] = G
        for (s, a), G_t in ep_Q.items():
            self._Q_cnt[s, a] += 1
            self._Q_hat[s, a] += (G_t - self._Q_hat[s, a]) / self._Q_cnt[s, a]
        for s in set(s for s, a, _ in self._buf):
            pi_s = _softmax(self.theta[s, :])
            V_s  = float(np.dot(pi_s, self._Q_hat[s, :]))
            self.theta[s, :] += self.lr * (self._Q_hat[s, :] - V_s)
    def get_policy(self):
        return np.vstack([_softmax(self.theta[s, :]) for s in range(self.n_states)])
    def get_value_estimate(self):
        V = np.zeros(self.n_states)
        for s in range(self.n_states):
            V[s] = float(np.dot(_softmax(self.theta[s, :]), self._Q_hat[s, :]))
        return V

print('NPG (unconstrained) ...')
npg_results = multi_run(
    lambda rng: NaturalPGRef(rng, GAMMA, lr=0.01), N_RUNS)

In [ ]:
cmp = {'TRPO (delta=0.01)': trpo_results, 'NPG (unconstrained)': npg_results}

fig = plot_multi_curves(cmp, metric='policy_err',
                         title='Policy Eval Error — TRPO vs NPG')
save_figure(fig, '09_policy_err')
plt.show()

## 4. KL Divergence and Step Size History

Track how the constraint behaves during training. The line search should keep
KL <= delta at every episode.

In [ ]:
# Train a diagnostic agent (single seed) to inspect KL and alpha history
diag_agent = TRPOAgent(np.random.default_rng(42), GAMMA, delta=DELTA)
diag_env   = GridWorldEnv(np.random.default_rng(43))
for ep in range(N_EPS):
    run_one_episode(diag_agent, diag_env, MAX_ST)

kl_arr    = np.array(diag_agent.kl_history)
alpha_arr = np.array(diag_agent.alpha_history)
eps_range = np.arange(1, len(kl_arr) + 1)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(eps_range, kl_arr, linewidth=0.7, color='steelblue')
axes[0].axhline(DELTA, color='red', linestyle='--', label=f'delta={DELTA}')
axes[0].set_ylabel('Actual KL')
axes[0].set_title('KL divergence per episode (constraint satisfied when <= delta)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(eps_range, alpha_arr, linewidth=0.7, color='darkorange')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Accepted step size alpha')
axes[1].set_title('Step size accepted by line search')
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
save_figure(fig, '09_kl_alpha_history')
plt.show()

pct_satisfied = (kl_arr <= DELTA + 1e-9).mean() * 100
print(f'KL constraint satisfied: {pct_satisfied:.1f}% of episodes')
print(f'Mean KL when satisfied:  {kl_arr[kl_arr > 0].mean():.5f}')

## 5. Delta Sensitivity

Larger delta = larger allowed steps = faster but riskier updates.
Smaller delta = more conservative = slower but more stable.

In [ ]:
delta_values = [0.001, 0.01, 0.05, 0.1]
delta_results = {}

for d in delta_values:
    res = multi_run(
        lambda rng, d=d: TRPOAgent(rng, GAMMA, delta=d), n_runs=N_RUNS)
    delta_results[f'delta={d}'] = res
    print(f'delta={d:<6}  final_policy_err={res["policy_err"][:, -1].mean():.4f}')

fig = plot_multi_curves(delta_results, metric='policy_err',
                         title='TRPO — Delta (KL radius) Sensitivity')
save_figure(fig, '09_delta_sweep')
plt.show()

## 6. Final Policy

In [ ]:
pi_trpo = diag_agent.get_policy()
V_trpo  = diag_agent.get_value_estimate()
print(f'TRPO  policy_err={policy_eval_error(pi_trpo):.4f}')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_vstar_heatmap(ax=axes[0])
plot_value_heatmap(V_trpo, title='TRPO  Q-estimated V(s)', ax=axes[1])
plot_policy_arrows(pi_trpo, title='TRPO  greedy policy', ax=axes[2])
fig.suptitle('TRPO — 1500 episodes, seed 42', fontsize=12, y=1.02)
fig.tight_layout()
save_figure(fig, '09_final_policy')
plt.show()

In [ ]:
final_pol = {k: v['policy_err'][:, -1].mean() for k, v in cmp.items()}
for k, v in final_pol.items():
    print(f'{k:<25}  policy_err={v:.4f}')
fig = plot_summary_bar(final_pol, title='Final Policy Eval Error — TRPO vs NPG')
save_figure(fig, '09_summary_bar')
plt.show()

## Discussion

### What TRPO adds over NPG
NPG computes the natural gradient direction and steps with a fixed learning
rate. If the step size is too large, the policy changes drastically, Q estimates
become invalid, and performance collapses. TRPO prevents this by:

1. Computing the *maximum* step that keeps KL <= delta (from the quadratic
   approximation).
2. Verifying the actual KL (exact computation) and shrinking if needed.

This makes TRPO more robust across different environments and hyperparameters.

### Tabular vs deep RL
In deep RL, F is a huge matrix (parameter space is high-dimensional) and
computing F^{-1} g requires the **conjugate gradient** algorithm with
Hessian-vector products. In the tabular case we have a closed form
(F_s^{-1} g_s = A_s), which makes the implementation exact and efficient.

### Proximal Policy Optimisation (PPO)
PPO simplifies TRPO by replacing the hard KL constraint with a soft clipping
of the probability ratio r = pi_new / pi_old. It achieves similar monotonic
improvement guarantees in practice without requiring line search, and is the
most widely used policy gradient method in deep RL today.

### Connection to the full algorithm family

```
REINFORCE  ->  NPG  ->  TRPO  ->  PPO
  SGD         nat. grad  constrained   clipped surrogate
```

Each step adds a correction for a specific failure mode of the previous method.